# Exercise: JSON Data Contracts

## Overview

A **data contract** is a formal agreement that defines the structure, rules, and expectations for a dataset. In this exercise you will write a JSON data contract for a provided student grades dataset, then validate the dataset against your contract using Python.

Data contracts are used in data engineering and analytics pipelines to ensure data quality, catch errors early, and create a shared understanding between data producers and consumers.

---

## The Dataset

You have been provided with `student_grades.csv`. Open the file and examine its contents before writing your contract. The dataset has four columns, all of which are required — no row should have a missing value in any column.

- **Student Name** — full name of the student (`string`)
- **Class** — the course the student is enrolled in (`string`, restricted values)
- **Semester** — the semester in which the class was taken (`string`, restricted values)
- **Grade** — the student's numeric grade (`integer`, restricted range)

**Approved values for `Class`:**
- Calculus 1
- English 101
- Python 101
- Intro to Databases

**Approved values for `Semester`:**
- Fall
- Spring

**Approved range for `Grade`:** 60 to 100 (inclusive)

---

## Part 1 — Write the JSON Data Contract

Create a file named `student_grades_contract.json`. Your contract must include the following four sections.

**1a. Contract Metadata**  
Include a top-level `contract` section with:
- `name` — the name of the dataset
- `version` — use `"1.0.0"`
- `description` — one sentence describing the dataset

**1b. Source Definition**  
Include a `source` section that describes the file:
- `format` — the file type
- `delimiter` — the character separating values
- `has_header` — whether the first row is a header (`true` or `false`)

**1c. Schema Definition**  
Include a `schema` section with a `fields` object. For each of the four columns define:
- `type` — the data type (`string` or `integer`)
- `nullable` — whether empty values are allowed (all columns are required)
- `allowed_values` — for `Class` and `Semester`, list the only accepted values
- `min_value` and `max_value` — for `Grade`, define the acceptable numeric range

**1d. Quality Rules**  
Include a `quality_rules` section with:
- `no_duplicate_rows` — set to `true`
- `min_row_count` — the minimum number of rows the file must contain (use `1`)

### Contract Skeleton

Use this structure as your starting point and fill in all the values:

```json
{
  "contract": {
    "name": "...",
    "version": "...",
    "description": "..."
  },
  "source": {
    "format": "...",
    "delimiter": "...",
    "has_header": ...
  },
  "schema": {
    "fields": {
      "Student Name": {
        "type": "...",
        "nullable": ...
      },
      "Class": {
        "type": "...",
        "nullable": ...,
        "allowed_values": [...]
      },
      "Semester": {
        "type": "...",
        "nullable": ...,
        "allowed_values": [...]
      },
      "Grade": {
        "type": "...",
        "nullable": ...,
        "min_value": ...,
        "max_value": ...
      }
    }
  },
  "quality_rules": {
    "no_duplicate_rows": ...,
    "min_row_count": ...
  }
}
```

---

## Part 2 — Validate the Dataset in Python

The following Python cells will validate your data contract against the student_grades.csv file. You do not need to do anything besides run the code to ensure you contract works. The contract steps are provided below simply for clafication:

1. Load the JSON contract using the `json` library
2. Load the CSV file using 'pandas'
3. Check that all required columns are present
4. Check that no column contains null or empty values
5. Check that all `Class` values match the allowed list in your contract
6. Check that all `Semester` values match the allowed list in your contract
7. Check that all `Grade` values are integers within the `min`/`max` range defined in your contract
8. Check for duplicate rows

For each check, a clear `PASS` or `FAIL` result will be printed. If a check fails, the rows that caused the failure will be printed.

---

## Deliverables

Submit the following file:

- `student_grades_contract.json` — your completed data contract

>  Ensure you use this exact naming convention or the validation script may not work.

---

## Tips

- Read the entire dataset before writing your contract — look at the actual values in each column.
- Your contract is just a structured JSON file. There is no single correct format, but it must contain all the required fields listed above.
- Test your validation script on the provided dataset — there should be **4 failed rows** if done correctly.

In [12]:
import pandas as pd
import json

# ── Load Contract ──────────────────────────────────────────────────────────────
with open("student_grades_contract.json", "r") as f:
    raw_contract = json.load(f)

contract = {
    "columns":           raw_contract["schema"]["expected_columns"],
    "allowed_classes":   raw_contract["schema"]["fields"]["Class"]["allowed_values"],
    "allowed_semesters": raw_contract["schema"]["fields"]["Semester"]["allowed_values"],
    "grade_min":         raw_contract["schema"]["fields"]["Grade"]["min_value"],
    "grade_max":         raw_contract["schema"]["fields"]["Grade"]["max_value"],
}

# ── Load Data ──────────────────────────────────────────────────────────────────
df = pd.read_csv("student_grades.csv")
print(f"Rows loaded: {len(df)}")
df.head()

# ── Check columns ──────────────────────────────────────────────────────────────
missing = [c for c in contract["columns"] if c not in df.columns]
if missing:
    print(f"FAIL — Missing columns: {missing}")
else:
    print("PASS — All expected columns present")

# ── Check for nulls ────────────────────────────────────────────────────────────
null_counts = df[contract["columns"]].isnull().sum()
if null_counts.sum() == 0:
    print("PASS — No null values found")
else:
    print("FAIL — Null values found:")
    print(null_counts[null_counts > 0])


Rows loaded: 100
PASS — All expected columns present
FAIL — Null values found:
Semester    1
dtype: int64


In [13]:
## check allowed values
bad_classes = df[~df["Class"].isin(contract["allowed_classes"])]
bad_semesters = df[~df["Semester"].isin(contract["allowed_semesters"])]

if bad_classes.empty:
    print("PASS — All Class values are valid")
else:
    print(f"FAIL — {len(bad_classes)} invalid Class value(s):")
    print(bad_classes[["Student Name", "Class"]])

if bad_semesters.empty:
    print("PASS — All Semester values are valid")
else:
    print(f"FAIL — {len(bad_semesters)} invalid Semester value(s):")
    print(bad_semesters[["Student Name", "Semester"]])

FAIL — 1 invalid Class value(s):
    Student Name    Class
85  Andrew Green  Dancing
FAIL — 1 invalid Semester value(s):
          Student Name Semester
84  Stephanie Martinez      NaN


In [14]:
## check grade range
bad_grades = df[(df["Grade"] < contract["grade_min"]) | (df["Grade"] > contract["grade_max"])]

if bad_grades.empty:
    print(f"PASS — All grades are between {contract['grade_min']} and {contract['grade_max']}")
else:
    print(f"FAIL — {len(bad_grades)} out-of-range grade(s):")
    print(bad_grades[["Student Name", "Grade"]])

FAIL — 2 out-of-range grade(s):
      Student Name  Grade
59     Carol Perez      2
92  Dorothy Torres    105


In [15]:
checks = {
    "No missing columns":   len(missing) == 0,
    "No null values":        null_counts.sum() == 0,
    "Valid Class values":    bad_classes.empty,
    "Valid Semester values": bad_semesters.empty,
    "Grades in range":       bad_grades.empty,
}

failed_records = {
    "Valid Class values":    bad_classes,
    "Valid Semester values": bad_semesters,
    "Grades in range":       bad_grades,
}

print("\n--- Validation Summary ---")
for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {check}")
    if not passed and check in failed_records:
        records = failed_records[check]
        if not records.empty:
            print(records.to_string(index=False))
            print()

overall = all(checks.values())
print('Validation Complete')


--- Validation Summary ---
  [PASS] No missing columns
  [FAIL] No null values
  [FAIL] Valid Class values
Student Name   Class Semester  Grade
Andrew Green Dancing   Spring     66

  [FAIL] Valid Semester values
      Student Name      Class Semester  Grade
Stephanie Martinez Python 101      NaN     85

  [FAIL] Grades in range
  Student Name              Class Semester  Grade
   Carol Perez Intro to Databases   Spring      2
Dorothy Torres Intro to Databases     Fall    105

Validation Complete
